# <span class="motutor-highlight motutor-id_dhad0ug-id_jdvlv8k"><i></i>本地部署DeepSeek模型</span>



本课程旨在帮助学生掌握 DeepSeek 模型的本地部署技术，包括硬件选择、量化方案、推理加速和 Web 部署等内容。通过理论学习和实践操作，学生将能够独立完成 DeepSeek 模型的部署。


本课程分为四个主要模块：

1. **不同硬件配置下的 DeepSeek 模型部署选择**
2. **模型量化方案对比**
3. **模型推理加速技术**
4. **DeepSeek 模型 Web 部署实践**


## 1. <span class="motutor-highlight motutor-id_8fh3abq-id_um600bk"><i></i>不同硬件配置下的 DeepSeek 模型部署选择</span>

当谈到 DeepSeek 模型的部署时，硬件配置是一个不可忽视的重要因素。那什么是合适的硬件配置呢？    

硬件配置是指运行模型所需的计算机硬件资源，包括 GPU、CPU、内存等。   
这些资源的选择直接影响着模型的运行效率和性能表现。   
因此，选择合适的硬件配置对于模型的成功部署至关重要。   


### 1.1 <span class="motutor-highlight motutor-id_zt9uad6-id_b70w13h"><i></i>硬件需求分析</span>

#### 1.1.1 模型规模与硬件需求

DeepSeek 模型的不同版本对硬件要求不同：





| 模型版本 | 参数量 | CPU | 内存 | 硬盘 | 显卡 |
|----------|--------|-----|------|------|------|
| DeepSeek-R1-1.5B | 15 亿 | 4 核（推荐Intel/AMD多核处理器） | 8GB+ | 3GB+（模型文件约1.5-2GB） | 非必需（纯CPU推理），若需 GPU 加速，建议 4GB+ 显存（如 GTX 1650） |
| DeepSeek-R1-7B | 70 亿 | 8 核以上 | 8 GB+ | 8GB+（模型文件约 4-5 GB） | 推荐 8GB+ 显存（如 RTX 3070/4060） |
| DeepSeek-R1-14B | 140 亿 | 12 核以上 | 16GB+ | 15GB+ | 16GB+ 显存（如 RTX 4090 或 A5000） |
| DeepSeek-R1-32B | 320 亿 | 16 核以上（如 AMD Ryzen 9 或 Intel i9） | 32GB+ | 30GB+ | 24GB+ 显存（如 A100 40GB 或双卡 RTX 3090） |
| DeepSeek-R1-70B | 700 亿 | 32 核以上（服务器级 CPU） | 128GB+ | 70GB+ | 多卡并行（如 2×A100 80GB 或 4×RTX 4090） |
| DeepSeek-R1-671B | 6710 亿 | 64 核以上（服务器集群） | 512GB+ | 300GB+ | 多节点分布式训练（如 8×A100/H100） | 


#### 1.1.2 <span class="motutor-highlight motutor-id_a0zukjz-id_gjq4rvv"><i></i>模型权重参数类型对比</span>

在深度学习模型中，权重参数是模型训练过程中不断调整的核心部分。不同的权重参数类型主要是指这些参数在计算机中存储和处理的数值格式。以下是几种常见的权重参数类型及其原理：
1. **FP32 (单精度浮点)**
   - **原理**：FP32 使用 32 位来表示一个浮点数，其中 1 位用于符号，8 位用于指数，23 位用于尾数。这种格式可以表示非常大的数值范围和较高的精度。
   - **优点**：计算精度高，适用于需要高精度计算的场景。
   - **缺点**：显存占用大，计算速度相对较慢。

2. **FP16 (半精度浮点)**
   - **原理**：FP16 使用 16 位来表示一个浮点数，其中 1 位用于符号，5 位用于指数，10 位用于尾数。相比 FP32，FP16 的数值范围和精度有所降低，但显存占用减少一半。
   - **优点**：显存占用较少，计算速度较快，精度损失较小。
   - **缺点**：在某些高精度要求的任务中可能不够用。

3. **BF16 (Brain 浮点)**
   - **原理**：BF16 也是一种 16 位浮点数格式，但它的指数部分与 FP32 相同（8位），尾数部分只有 7 位。这使得 BF16 在数值范围上与 FP32 相同，但精度较低。
   - **优点**：数值稳定性好，适用于大模型训练和需要数值稳定的场景。
   - **缺点**：精度不如 FP32，但在大多数训练任务中足够用。

4. **INT8 (8 位定点)**
   - **原理**：INT8 使用 8 位整数来表示权重参数。通过量化技术，将浮点数映射到 8 位整数范围内，从而大幅减少显存占用和计算量。
   - **优点**：显存占用低，计算速度快，适用于对精度要求一般的任务。
   - **缺点**：精度损失较大，可能影响模型性能。

5. **INT4 (4 位定点)**
   - **原理**：INT4 使用 4 位整数来表示权重参数，是一种更为极端的量化方式。虽然显存占用和计算量进一步减少，但精度损失也更为显著。
   - **优点**：显存占用最低，计算速度最快，适用于对精度要求不高的任务。
   - **缺点**：精度损失较大，可能严重影响模型性能。

不同的权重参数类型在显存占用、计算速度和精度之间存在权衡，选择合适的权重参数类型需要根据具体的应用场景和硬件资源来决定，可参考以下表格进行评估：

| 参数类型 | 精度 | 显存占用 | 特点 | 适用场景 |
|----------|------|----------|------|----------|
| **FP32 (单精度浮点)** | 32 位浮点数 | 每个参数 4 字节 | - 最高的计算精度<br>- 显存占用最大<br>- 推理速度最慢 | - 模型训练阶段<br>- 需要极高精度的科研场景<br>- 硬件资源充足的环境 |
| **FP16 (半精度浮点)** | 16 位浮点数 | 每个参数 2 字节 | - 精度损失较小<br>- 显存占用中等<br>- 推理速度适中 | - 对精度要求较高的任务<br>- 显存资源充足的环境<br>- 标准生产环境部署 |
| **BF16 (Brain 浮点)** | 16 位浮点数(特殊格式) | 每个参数 2 字节 | - 相比 FP16 有更大的动态范围<br>- 数值稳定性好<br>- 训练时梯度更稳定 | - 大模型训练<br>- 需要数值稳定的场景<br>- 支持 BF16 的新型硬件 |
| **INT8 (8 位定点)** | 8 位整数 | 每个参数 1 字节 | - 精度损失可控<br>- 显存占用较低<br>- 推理速度较快 | - 对精度要求一般的任务<br>- 显存资源受限环境<br>- 需要加快推理速度时 |
| **INT4 (4 位定点)** | 4 位整数 | 每个参数 0.5 字节 | - 精度损失较大<br>- 显存占用最低<br>- 推理速度最快 | - 对精度要求不高的任务<br>- 显存资源严重受限<br>- 追求最快推理速度 |


#### 1.1.3 <span class="motutor-highlight motutor-id_7750xr1-id_h55kprh"><i></i>关键硬件指标</span>

| 硬件指标 | 重要性 | 影响因素 |
|----------|--------|----------|
| **GPU显存** | 最关键的指标 | - 模型参数数量<br>- 量化方案选择<br>- 批处理大小 |
| **CPU核心数** | 影响数据处理能力 | - 数据预处理速度<br>- 系统响应能力<br>- 多任务处理能力 | 
| **内存大小** | 影响系统稳定性 | - 模型加载时间<br>- 数据处理能力<br>- 系统稳定性 | 
| **存储空间** | 影响模型加载速度 | - 模型文件大小<br>- 数据缓存需求<br>- 系统响应速度 | 

### 1.2 <span class="motutor-highlight motutor-id_og5ybz9-id_5rbono2"><i></i>常见硬件配置方案</span>

在实际部署 DeepSeek 模型时，我们需要根据不同的使用场景选择合适的硬件配置。   
那如何选择合适的配置呢？    

我们可以根据模型规模、使用场景和预算等因素，将硬件配置分为入门级、中端和高端三个层次。   
每个层次都有其特定的适用场景和性能表现。   


| 模型版本 | 量化方案 | 适用场景 | GPU配置 | CPU配置 | 内存配置 | 存储配置 |
|----------|----------|----------|---------|---------|----------|----------|
| DeepSeek-7B | FP16 | 个人开发测试、小规模应用 | RTX 3060 (12GB)+ | i5/Ryzen 5 (8核) | 16GB | 500GB SSD |
| DeepSeek-7B | INT8 | 资源受限环境、入门级部署 | GTX 1660 (6GB)+ | i5/Ryzen 5 (8核) | 16GB | 500GB SSD |
| DeepSeek-7B | INT4 | 极低资源环境、轻量级应用 | GTX 1650 (4GB)+ | i5/Ryzen 5 (8核) | 16GB | 500GB SSD |
| DeepSeek-14B | FP16 | 中等规模应用、专业开发 | RTX 3090 (24GB)+ | i7/Ryzen 7 (12核) | 32GB | 500GB SSD |
| DeepSeek-14B | INT8 | 平衡性能与资源 | RTX 3060 (12GB)+ | i7/Ryzen 7 (12核) | 32GB | 500GB SSD |
| DeepSeek-14B | INT4 | 中等资源环境 | RTX 2060 (8GB)+ | i7/Ryzen 7 (12核) | 32GB | 500GB SSD |
| DeepSeek-67B | FP16 | 大规模生产、高性能要求 | A100 (80GB)+ | i9/Ryzen 9 (16核) | 64GB | 500GB SSD |
| DeepSeek-67B | INT8 | 高端应用、资源优化 | RTX 4090 (24GB)+ | i9/Ryzen 9 (16核) | 64GB | 500GB SSD |
| DeepSeek-67B | INT4 | 高端资源受限环境 | RTX 3090 (24GB)+ | i9/Ryzen 9 (16核) | 64GB | 500GB SSD |


### 1.3 <span class="motutor-highlight motutor-id_9vofvib-id_31jpx70"><i></i>硬件性能评估</span>

在实际部署过程中，我们需要对硬件性能进行监控和评估。   
那如何评估硬件性能呢？    

我们可以通过以下几个方面来评估：

#### 1.3.1 显存使用分析

GPU 资源监控代码需要在存在GPU的环境中运行，下面给出代码样例。

```python
import torch
import psutil
import GPUtil

def analyze_gpu_memory():
    # 获取GPU信息
    gpus = GPUtil.getGPUs()
    for gpu in gpus:
        print(f"GPU: {gpu.name}")
        print(f"显存总量: {gpu.memoryTotal}MB")
        print(f"显存使用: {gpu.memoryUsed}MB")
        print(f"显存剩余: {gpu.memoryFree}MB")

# 运行分析
analyze_gpu_memory()

```


#### 1.3.2 系统资源监控
运行以下代码进行当前平台系统资源的监控。

In [ ]:
import psutil

def monitor_system_resources():
    # CPU使用率
    cpu_percent = psutil.cpu_percent(interval=1)
    print(f"CPU使用率: {cpu_percent}%")
    
    # 内存使用情况
    memory = psutil.virtual_memory()
    print(f"内存使用率: {memory.percent}%")
    print(f"可用内存: {memory.available / (1024**3):.2f}GB")

# 运行监控
monitor_system_resources()

### 1.4 <span class="motutor-highlight motutor-id_1ott6zl-id_lz3pgsu"><i></i>硬件选择建议</span>

在选择硬件配置时，我们需要考虑多个因素。   
那如何做出最佳选择呢？    

1. **个人开发测试**
   - 建议选择入门级配置
   - 重点考虑性价比
   - 可以接受较慢的推理速度

2. **生产环境部署**
   - 建议选择中端或高端配置
   - 注重稳定性和性能
   - 需要考虑扩展性

下方给出成本分析的样例代码，大家可以根据自己关注的指标修改代码以选择适合自己的配置




In [ ]:

def calculate_cost_efficiency(gpu_price, performance_score):
    """
    计算硬件配置的成本效益比
    
    参数:
    gpu_price: GPU价格（元）
    performance_score: 性能评分（1-10）
    
    返回:
    cost_efficiency: 成本效益比
    """
    cost_efficiency = performance_score / (gpu_price / 10000)
    return cost_efficiency

# 示例计算
gpu_configs = {
    "RTX 3060": {"price": 3000, "performance": 6},
    "RTX 3080": {"price": 6000, "performance": 8},
    "A5000": {"price": 15000, "performance": 9}
}

for gpu, specs in gpu_configs.items():
    efficiency = calculate_cost_efficiency(specs["price"], specs["performance"])
    print(f"{gpu}的成本效益比: {efficiency:.2f}")


## 2. DeepSeek 模型量化方案对比

### 2.1 <span class="motutor-highlight motutor-id_tqcudfw-id_jyocyvq"><i></i>模型量化基础</span>

#### 2.1.1 什么是模型量化？
在我们的深度学习世界里，模型就像是一个聪明的大脑，可以做很多复杂的事情，比如图像识别、语音识别。  
但是呢，这个大脑有时候有点 “胖”，需要很多的 “空间”（存储）和 “能量”（计算资源）。
>想象一下，如果要把这个大脑装到一个小小的手机或者其他小型设备里，就会很困难。 

>模型量化就是一种神奇的技术，它可以让这个 “大脑” 变得更 “瘦”，也就是减少它需要的存储和计算资源，同时还能尽量保持它的聪明程度（性能）。今天我们就来一起学习模型量化这个有趣的技术。



<div class='insertContainerBox column'>
<div class='insertItem' align=center><img src="https://imgbed.momodel.cn/hv/202504240924501.jpeg" width="800px"/></div>
</div>


#### 2.1.2 <span class="motutor-highlight motutor-id_57y9713-id_gbj4ugq"><i></i>基本概念</span>
定义：模型量化简单来说，就是把模型里的一些数字（像权重和激活值，这些就像是模型做决策的重要依据），用一些更小、更简单的数据类型来代替。  
比如说，原来是用很精确的 32 位浮点数，现在用精度低一些的 8 位整数或者 16 位浮点数来表示。  

目的：举个例子，假如我们有一个特别大的模型，它要占用很大的存储空间，就像一个大房子需要很多地方来存放。而且计算的时候也很慢，就像一个人做很多复杂数学题要花很长时间。  
通过模型量化，我们可以让这个模型变得小一些，存储起来不占那么多地方，计算起来也能更快，这样就能轻松地放到一些小设备里，比如智能手表、智能摄像头这些嵌入式设备。  

#### 2.1.3 <span class="motutor-highlight motutor-id_kj8h68p-id_ecp7hjg"><i></i>量化原理</span>
线性量化的原理可以这样通俗理解：

```
想象我们有一大箱不同尺寸的积木，这些积木的尺寸都有非常精确的测量值，这就好比模型中的浮点数。现在，我们要把这些积木放到一个小盒子里，但小盒子装不下这么多不同尺寸的积木，而且处理起来也很麻烦。 

于是，我们决定用一些简单规则来重新整理这些积木。线性量化就像是给这些积木制定一个新的 “衡量标准”。我们先找一个固定的比例（类似缩放因子），把所有积木的尺寸都按照这个比例进行缩小或者放大。比如，原来有一块积木长度是 10.5 厘米，我们规定一个比例，让它缩小到原来的十分之一，那就变成 1.05。  

然后，我们再设定一个固定的数值（类似零点偏移），把调整后的尺寸加上这个固定数值。假如固定数值是 0，那这个积木现在就是 1.05 。但为了让它更简单，我们只保留整数部分，这块积木就变成 1 。这样，所有积木都按照这个规则变成了更简单的整数尺寸。  

在需要使用这些积木的时候，也就是反量化过程，我们再按照相反的规则，把这个整数尺寸变回原来大概的尺寸，让它能继续发挥作用。 
```
通过这种方式，我们把复杂多样的 “积木尺寸”（浮点数）用简单统一的方式（整数）表示出来，虽然会损失一点点精度，但在模型中就可以更方便、快速地进行处理啦，这就是线性量化的基本原理。

#### 2.1.4 <span class="motutor-highlight motutor-id_brjy3l3-id_ct18ssi"><i></i>量化方法分类</span>
**训练后量化（PTQ, Post-Training Quantization）**     
- 定义：模型已经训练好了，这时候我们直接对它进行量化，不用再重新训练模型。  
- 优点：就像我们已经做好了一个蛋糕，现在只是给它包装一下，很简单很快，也不需要额外的材料（训练数据）和时间。  
- 缺点：但是这样包装可能不太完美，蛋糕可能会有一点点变形，也就是模型的性能可能会下降一些。  
- 常见方法：静态量化就是在开始推理（模型做预测）之前，就把量化的参数都算好；动态量化是在推理的过程中，一边推理一边动态地计算量化参数。  

**量化感知训练（QAT, Quantization-Aware Training）**      
- 定义：在模型训练的过程中，我们就开始模拟量化操作，让模型提前适应量化带来的信息损失。  
- 优点：这样训练出来的模型，就像是经过特殊训练的运动员，能更好地适应量化，模型性能下降得就比较少。  
- 缺点：但是这就需要我们重新训练模型，就像运动员要重新进行很多次训练一样，会花更多的时间和计算资源。  

#### 2.1.5 <span class="motutor-highlight motutor-id_3sha2g6-id_rxuojdq"><i></i>量化的影响</span>
- 计算效率提升：我们用电脑做计算的时候，整数运算就比浮点数运算要快很多。就像跑步，整数运算就像是短跑选手，跑得很快；浮点数运算就像是长跑选手，速度慢一些。模型量化用低精度的数据类型，也就是用整数运算比较多，这样计算速度就大大提高。  
- 内存占用减少：低精度的数据类型就像小盒子，占的空间小；高精度的数据类型就像大盒子，占的空间大。把模型里的数据类型换成低精度的，就像把大盒子换成小盒子，模型需要的存储空间就变小。  
- 性能损失：量化就像给一幅画加上了一些模糊效果，会损失一些信息，所以模型的性能可能会下降。  

#### 2.1.6 4bit 与 8bit 量化

| 特性            | 4bit 量化方案                                      | 8bit 量化方案                                      |
|-----------------|--------------------------------------------------|--------------------------------------------------|
| **量化原理**    | 使用 4 位整数表示原始浮点数，映射并通过反量化恢复精度 | 使用 8 位整数表示原始浮点数，映射并通过反量化恢复精度 |
| **精度损失**    | 理论精度损失约 75%，一般任务影响较小，复杂任务可能较大 | 理论精度损失约 50%，一般任务影响很小，复杂任务可控   |
| **性能提升**    | 显存占用减少约 75%，推理速度提升约 2-3 倍，能耗降低约 60-70% | 显存占用减少约 50%，推理速度提升约 1.5-2 倍，能耗降低约 40-50% |
| **优势**        | 资源节省，适合资源受限环境，部署门槛低，维护成本低 | 精度平衡，兼容性好，适用场景广泛，部署更稳定         |
| **局限性**      | 精度问题，部分操作不支持，使用限制较多             | 资源消耗较大，硬件要求高，性能提升有限               |
| **应用场景**    | 个人开发测试，边缘设备部署，资源受限环境           | 生产环境部署，性能要求较高，需要稳定性               |

### 2.2 <span class="motutor-highlight motutor-id_j2jfebb-id_hp0fqwg"><i></i>场景案例  </span>

假设我们要在一个智能安防摄像头中部署一个目标检测模型，用来识别画面中的人物、车辆等目标。  

<div class='insertContainerBox column'>
<div class='insertItem' align=center><img src="https://imgbed.momodel.cn/202502110910284.png" width="800px"/></div>
</div>

在这个场景下，摄像头的存储容量和计算能力都是有限的。如果直接使用未经量化的原始模型，可能会出现以下问题：  
- 存储问题：原始模型文件体积较大，很快就会占满摄像头的存储空间，导致无法持续存储视频数据以及模型相关信息。  
- 计算问题：由于原始模型计算量较大，摄像头的芯片在处理视频帧时会变得非常缓慢，无法实时地检测出目标，导致延迟过高，影响安防效果。  

现在我们对这个目标检测模型进行量化处理：  
- **采用训练后量化（PTQ）中的静态量化方法**：我们拿到已经训练好的模型，通过一系列的计算确定量化参数，比如缩放因子和零点偏移等。将模型中的权重和激活值按照量化规则转换为 8 位整数。经过量化后，模型的存储大小显著降低，原本可能需要几十 MB 的模型，现在可能只需要几 MB。在计算方面，整数运算的速度提升明显，摄像头能够快速处理视频帧，实现实时的目标检测。虽然模型性能略有下降，比如检测的准确率从 95% 下降到了 92%，但在可接受范围内，满足了安防摄像头实时性和存储容量的要求。  

- **采用量化感知训练（QAT）**：在模型训练阶段，我们就引入量化模拟操作。在训练过程中，模型逐渐适应量化带来的信息损失。经过这样训练后的模型，在量化为 8 位整数后，性能损失极小，检测准确率可能仅下降到 94%。而且在存储和计算效率提升方面与 PTQ 类似，同样大幅减少了存储需求，提高了计算速度。  

通过这个案例可以看到，模型量化在实际应用场景中，能够有效地解决设备资源有限的问题，并且不同的量化方法对模型性能的影响也有所不同。  


### 2.3 总结 

模型量化是将模型中的浮点数参数用低精度数据类型近似表示的技术，目的是降低存储成本、减少内存访问带宽和加速计算。主要量化方法有训练后量化（PTQ）和量化感知训练（QAT），PTQ 简单快速但可能导致性能较大下降，QAT 能更好地保持性能但需要重新训练模型。  

## 3. <span class="motutor-highlight motutor-id_3pn1fom-id_12hlx1m"><i></i>DeepSeek 模型推理加速技术 </span> 

在如今的人工智能时代，大语言模型变得越来越强大，它们能做很多神奇的事情，比如像智能聊天机器人和自动生成文章。  
但是，这些模型在运行的时候常常会遇到速度慢的问题，就好比一辆车开得不够快，不能及时把我们需要的结果 “送” 过来。这时候，模型加速服务就显得尤为重要。  
今天我们要学习的 vllm 和 TGI 就是两款非常厉害的模型加速服务工具，它们能让大语言模型跑得更快，为我们带来更好的体验。  


### 3.1 <span class="motutor-highlight motutor-id_m1tgztl-id_6xcfj4a"><i></i>vLLM</span>

vLLM（Virtual Large Language Model） 是由斯坦福大学开发的一个高效推理框架，专注于通过虚拟化的内存管理来提高推理效率。  
它就像是一个给大语言模型提速的 “小助手”，能让模型运行得又快又好，帮我们减少等待结果的时间。  
比如说，我们用一个大语言模型写作文，如果没有 vLLM，可能要等半分钟才能得到结果，有了 vLLM，可能十几秒就能拿到。

<div class='insertContainerBox column'>
<div class='insertItem' align=center><img src="https://imgbed.momodel.cn/hv/202504241030731.png" width="600px"/></div>
</div>

中文网：https://vllm.hyper.ai/

#### 3.1.1 核心特性
- PagedAttention 算法：  
这个算法类似操作系统的虚拟内存。想象一下，我们的电脑内存就像一个大仓库，里面存放着很多东西。  
在传统的注意力机制中，东西放得乱七八糟，出现了内存碎片化的问题，找东西就很麻烦。  
而 PagedAttention 算法就像是一个聪明的仓库管理员，它把注意力计算中的键值缓存(kv cache)进行分页管理，让仓库变得井井有条，这样找东西就快的多，也就是减少了内存开销，提高了内存利用率，推理过程自然就加速了。  
例如，当我们使用聊天机器人时，这个算法能让机器人更快地处理我们说的话。

- 支持并行解码：  
GPU 就像一个超级大工厂，里面有很多工人在工作。  
并行解码就是让很多工人同时工作，一起处理不同的请求。这样一来，模型就能同时处理好多任务，速度也就提升了。  
就像工厂里很多工人同时做不同的产品，生产效率就大大提高。

#### 3.1.2 <span class="motutor-highlight motutor-id_4lmrsui-id_5jaolfv"><i></i>优势</span>

1. **高性能**：凭借 PagedAttention 等技术，vLLM 在推理性能上领先，能提升吞吐量、降低延迟，满足高并发请求。

2. **内存管理高效**：优化内存使用，支持更大模型和更长上下文，减少 GPU 资源占用，降低部署成本。

3. **易用性**：与 Hugging Face Transformers 生态系统集成，提供 OpenAI 兼容 API，便于加载预训练模型和集成到应用。

4. **灵活性与可扩展性**：支持多种解码算法（如 Beam Search、Parallel Sampling）和分布式推理，还具备流式输出功能。

5. **开源与社区驱动**：开源特性允许用户自由查看、修改代码，社区驱动模式促进持续改进和更新。

#### 3.1.3 应用场景
vLLM 的高性能和高效率使其在众多 LLM 应用场景中具有广泛的应用前景，包括但不限于：

- 实时交互应用: 例如智能客服、聊天机器人、实时翻译等，这些应用对延迟 非常敏感，vLLM 的低延迟特性可以保证用户获得流畅的交互体验。
- 高吞吐量服务: 例如大规模 API 服务、内容生成平台等，这些场景需要高吞吐量 来支持大量的并发请求，vLLM 的高吞吐量可以显著降低服务成本，提升服务能力。
- 资源受限环境: 例如边缘设备、移动设备等，这些环境的计算资源和内存资源有限，vLLM 的低内存占用 特性使得在这些资源受限的环境中部署 LLM 应用成为可能。
- LLM 研究与开发: vLLM 作为一个高性能的推理引擎，可以为 LLM 的研究和开发提供强大的工具支持，加速新模型的验证和迭代过程。


### 3.2 <span class="motutor-highlight motutor-id_fhxudhl-id_jmktlh9"><i></i>TGI</span>
TGI （Text Generation Inference） 是 Hugging Face 推出的一个轻量级推理框架，专注于为大语言模型提供一种高效、简单的推理解决方案。  
TGI 侧重于通过 模型优化和负载均衡 来提升推理性能，适合大规模部署环境。
简单来说，它就像是给大语言模型装上了一个 “加速引擎”，能让模型在进行文本生成推理的时候速度更快、效率更高。  



#### 3.2.1 核心特性

- 优化的序列生成：TGI 针对序列生成任务进行了优化，使得模型在生成长文本时能够更高效地处理。  
- 负载均衡：TGI 在多机、多 GPU 场景下支持分布式推理，通过负载均衡提高多请求处理能力。
- 高可扩展性：TGI 通过优化网络通信和数据流管理，在分布式环境中表现出良好的扩展性，适合大规模模型部署。
- 针对序列生成任务的优化：TGI 在序列生成任务上进行了优化，在标准的生成任务中，推理速度较快，尤其是当序列长度较短或中等长度时，TGI 的推理速度可能会优于 vLLM。  
- 负载均衡的吞吐量优化：在多 GPU 和多任务请求的环境下，TGI 能够通过负载均衡机制，均衡多个 GPU 的计算压力，提升吞吐量。在大规模集群环境中，TGI 通过高效的调度和分配策略，实现了出色的吞吐量表现。



#### 3.2.2 <span class="motutor-highlight motutor-id_3rrscp1-id_w9o5iiu"><i></i>性能表现 </span> 

TGI 通过负载均衡和分布式推理来优化资源利用率，尤其适合大规模集群部署环境。  
- 分布式架构：TGI 在多机多 GPU 环境下，能够高效地进行资源分配和任务调度。其负载均衡机制确保了各个 GPU 之间的工作量平均分配，最大限度地提高了集群的计算效率。
- 内存优化：虽然 TGI 并没有 vLLM 那样复杂的缓存优化机制，但它通过精简的模型计算和序列生成优化，减少了内存占用，从而在短序列生成任务中表现出色。



#### 3.2.3 易用性

TGI  以轻量级、可扩展性为设计初衷，其灵活性主要体现在其分布式部署和简单的架构设计上。  
- 多任务环境支持：TGI 非常适合多机、多任务的部署场景。其分布式架构设计允许开发者根据任务需求自由调整部署规模，提供了良好的横向扩展能力。  
- 易于集成：TGI 框架与      Hugging Face 的生态深度集成，因此开发者可以很方便地在现有项目中集成 TGI，用于推理任务。而且，TGI 的易用性使其成为许多没有专门硬件优化需求的开发者首选。



#### 3.2.4 应用场景  
- 短文本生成：在短文本或中等长度的序列生成任务中，TGI 的性能极为优秀，适合自动化对话、生成代码片段等任务。
- 大规模部署：TGI 在分布式环境中的高可扩展性，使其非常适合需要多机集群部署的大规模推理任务，如大规模 API 服务。



## 4. <span class="motutor-highlight motutor-id_eodc7b7-id_2r01y4s"><i></i>DeepSeek 模型 Web 部署</span>

在自然语言处理领域，模型训练出来后，如何将其部署到实际应用场景中非常关键。比如我们常见的智能客服、智能写作助手等，背后都是模型部署在发挥作用。  
部署能让模型真正为用户提供服务，实现其价值。DeepSeek 模型具有独特的优势，它在大规模数据上进行训练，在语言理解和生成任务中表现出色。  
而 Ollama 是一款强大的模型部署工具，能帮助我们轻松地将 DeepSeek 模型部署到本地环境。



### 4.1 <span class="motutor-highlight motutor-id_76xrfo5-id_ez32t6h"><i></i>Ollama 简介</span>

- 基本概念和工作原理：Ollama 是一个用于管理和运行语言模型的工具，它通过容器化技术，将模型及其依赖环境打包在一起，方便在不同环境中部署和运行。  
例如，它可以在本地计算机上快速启动一个模型服务，让用户可以通过 API 进行调用。
- 对比优势：与其他模型部署工具相比，Ollama 具有更简单的安装和配置过程，对硬件资源的要求相对较低，并且支持多种模型格式。
- 主要功能和使用场景：主要功能包括模型的下载、加载、管理和推理。适用于本地开发、测试以及小型项目的模型部署。



<div class='insertContainerBox column'>
<div class='insertItem' align=center><img src="https://imgbed.momodel.cn/hv/20250619155402214.png" width="1200px"/></div>
</div>


#### 4.1.1 安装 Ollama

1. 确定系统 CPU 架构是 ARM 还是 AMD（X86）  
   使用 `lscpu` 或者 `uname -a` 查看自己 CPU 架构。

2. 根据 CPU 型号下载安装对应 Ollama 包，下载地址：[https://github.com/ollama/ollama/releases/](https://github.com/ollama/ollama/releases/)

   - x86_64 CPU 选择下载 `ollama-linux-amd64.tgz`
   - arm64 CPU 选择下载 `ollama-linux-arm64.tgz`

   《考虑多数 CPU 架构为 x86，以下按照 x86 架构进行介绍，AMD 架构可参考 [https://github.com/ollama/ollama/blob/main/docs/linux.md](https://github.com/ollama/ollama/blob/main/docs/linux.md)》

3. 解压：

   ```bash
   sudo tar -C /usr -xzf ollama-linux-amd64.tgz
   ```

4. 添加权限：

   ```bash
   chmod +x /usr/bin/ollama
   ```

5. 创建 Ollama 用户：

   ```bash
   useradd -r -s /bin/false -m -d /usr/share/ollama ollama
   ```

6. 启动：

   ```bash
   ollama serve
   ```

7. 配置 Ollama

-  编辑配置文件：

   ```bash
   vim /etc/systemd/system/ollama.service
   ```

   ```plaintext
   [Unit]
   Description=Ollama Service
   After=network-online.target

   [Service]
   ExecStart=/usr/bin/ollama serve
   User=ollama
   Group=ollama
   Restart=always
   RestartSec=3
   Environment="PATH=/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/root/bin"
   Environment="CUDA_VISIBLE_DEVICES=0,1"  # 代表让 Ollama 能识别到第几张显卡
   Environment="OLLAMA_SCHED_SPREAD=1"     # 这几张卡均衡使用
   Environment="OLLAMA_KEEP_ALIVE=-1"      # 模型一直加载, 不自动卸载
   Environment="OLLAMA_HOST=0.0.0.0"       # 配置远程访问
   Environment="OLLAMA_ORIGINS=*"          # 配置跨域请求
   Environment="OLLAMA_MODELS=/data/soft/ollama/.ollama/models"  # 配置 OLLAMA 的模型存放路径，默认路径是 /usr/share/ollama/.ollama/models/
   [Install]
   WantedBy=default.target
   ```

   注意：1. 远程和跨域访问务必配置 `OLLAMA_HOST` 和 `OLLAMA_ORIGINS` 参数。2. Ollama 默认端口为 11434。

-  创建文件并设置权限：

   ```bash
   sudo mkdir -p /data/soft/ollama/.ollama/models
   sudo chown -R ollama:ollama /data/soft/ollama/.ollama
   ```

8. 执行命令启动ollama

   ```bash
   sudo systemctl daemon-reload
   sudo systemctl start ollama
   ```
   
9. 测试是否启动成功：

   ```bash
   ollama --version
   ```


#### 4.1.2 <span class="motutor-highlight motutor-id_t1gur7p-id_y7lq7r0"><i></i>运行 Ollama</span>

- 模型列表：使用以下命令查看本地有哪些模型：
    ```bash
    ollama list
    ```
    如果模型出现在列表中，则说明模型存在于本地并可以使用。

- 运行模型：使用以下命令通过Ollama 调用 DeepSeek 模型：
    ```bash
    ollama run deepseek-model
    ```
    其中 `deepseek-model` 是模型的名称，确保模型文件已正确配置在 Ollama 的配置文件中。如果模型不在本地环境中，Ollama 会自动下载模型。  
    支持的模型列表参见链接  
    运行后就可以进行对话，模型会根据输入的问题生成相应的回答并输出到终端。  



#### 4.1.3 Ollama 运行自定义模型

- 以加载 GGUF 后缀模型为例：在 Ollama 中加载自定义 GGUF 类型文件可以按照以下步骤进行。
- 创建 Modelfile 文件：创建一个名为 Modelfile 的文件，在其中使用 FROM 指令指定要导入的本地模型文件路径。例如，如果你的 GGUF 模型文件是 vicuna-33b.Q4_0.gguf，那么在 Modelfile 中写入：
  ```plaintext
  FROM ./vicuna-33b.Q4_0.gguf
  ```
- 在 Ollama 中创建模型：使用 ollama create 命令根据 Modelfile 创建模型。在命令行中运行以下命令：
  ```bash
  ollama create example -f Modelfile
  ```
  这里的 example 是你为创建的模型指定的名称，你可以根据需要自行修改。-f 参数用于指定 Modelfile 的路径。
- 运行模型：模型创建完成后，使用 ollama run 命令来运行该模型。例如：
  ```bash
  ollama run example
  ```
  通过以上步骤，你就可以在 Ollama 中加载自定义的 GGUF 类型文件并运行相应模型。


### 4.2 <span class="motutor-highlight motutor-id_ftc74wx-id_8vci0r2"><i></i>OpenWebUI</span>

OpenWebUI 是一个基于 Web 的用户界面，它可以让我们更方便地与部署的模型进行交互，提供更友好的用户体验。



#### 4.2.1 OpenWebUI 简介

- 基本概念和特点：OpenWebUI 是一个开源的项目，它提供了可视化的界面，允许用户通过浏览器与模型进行交互。其特点包括易于使用、可定制性强等。例如，用户可以根据自己的需求调整界面布局和样式。
- 工作原理：它通过与后端的模型服务进行通信，将用户的输入传递给模型，并将模型的输出展示给用户。例如，当用户在浏览器中输入问题时，OpenWebUI 将问题发送到运行 DeepSeek 模型的服务器，服务器返回答案后，OpenWebUI 再将答案显示在页面上。
- 与其他工具的比较优势：与一些传统的模型交互方式相比，OpenWebUI 不需要用户掌握复杂的命令行操作，更适合普通用户和非技术人员使用。同时，它支持多模型集成，可以同时管理和使用多个不同的模型。





<div class='insertContainerBox column'>
<div class='insertItem' align=center><img src="https://imgbed.momodel.cn/hv/20250619155552832.gif" width="1200px"/></div>
</div>


#### 4.2.2 <span class="motutor-highlight motutor-id_baebzm3-id_tocx76h"><i></i>安装 OpenWebUI</span>

- 离线安装 docker：保姆级 docker 离线安装教程可参考 [此链接](https://zhuanlan.zhihu.com/p/578402141)。
- 联网服务器上下载、保存、上传、解压 docker 镜像，如果在联网环境中部署 OpenWebUI， 则通过第一步拉取镜像即可：
  1. 下载镜像：
     ```bash
     docker pull ghcr.io/open-webui/open-webui:main
     ```
  2. 保存镜像：
     ```bash
     docker save > open-webui.tar ghcr.io/open-webui/open-webui:main
     ```
  3. 拷贝到离线服务器并加载到 docker 中：
     ```bash
     docker load < open-webui.tar
     ```
  4. 验证是否加载成功：
     ```bash
     docker images
     ```
- 启动 open-webui：
  ```bash
  docker run -d -p 3000:8080 --add-host=host.docker.internal:host-gateway -v /data/ollama/open-webui:/app/backend/data -e OLLAMA_BASE_URL=http://127.0.0.1:11434 --name open-webui --restart always ghcr.io/open-webui/open-webui:main
  ```
- 登录：
  打开浏览器访问 `http://xxxxxx:3000/auth`。